In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B"

In [ ]:
# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer for a specific Gemma model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [ ]:
tokenizer.special_tokens_map

{'bos_token': '<bos>',
 'eos_token': '<eos>',
 'unk_token': '<unk>',
 'pad_token': '<pad>',
 'mask_token': '<mask>',
 'audio_token': '<|audio|>',
 'boa_token': '<|audio>',
 'boi_token': '<|image>',
 'eoa_token': '<audio|>',
 'eoc_token': '<channel|>',
 'eoi_token': '<image|>',
 'eot_token': '<turn|>',
 'escape_token': '<|"|>',
 'etc_token': '<tool_call|>',
 'etd_token': '<tool|>',
 'etr_token': '<tool_response|>',
 'image_token': '<|image|>',
 'soc_token': '<|channel>',
 'sot_token': '<|turn>',
 'stc_token': '<|tool_call>',
 'std_token': '<|tool>',
 'str_token': '<|tool_response>',
 'think_token': '<|think|>'}

In [ ]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
      (layers): ModuleList(
        (0-3): 4 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=1536, out_features=256, bias=False)
            (v_proj): Linear(in_features=1536, out_features=256, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
            (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
            (down_proj): Linear(in_features=6144, out_features=1536, bias=False)


In [ ]:
def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
generate_text("Hey how are you doing")

'Hey how are you doing? I hope your day is going well!\n\nHow have you been since our last call? If you were worried about something, we covered it. If you have any questions about your current situation, we can talk about'

In [ ]:
import torch
torch.tensor([tokenizer.encode("Hey how are you doing?")]).shape

torch.Size([1, 7])

In [ ]:
output = model(torch.tensor([tokenizer.encode("Hey how are you doing?")], device=device))

In [ ]:
output.logits[:,-1,:].shape

torch.Size([1, 262144])

In [ ]:
generate_text("How many letters in word banana?")

'How many letters in word banana?\n\nHow many letters in word banana?\n\nHow many letters in the word apple?\n\nWhich of these is not a sound?\n\nHow many syllables are in the word umbrella?\n\nHow many letters in the'

In [ ]:
import torch.nn.init as init

class GemmaForMCQs(torch.nn.Module):
    ID_TO_LABEL = {
        0: 'A',
        1: 'B',
        2: 'C',
        3: 'D',
        4: 'E'
    }

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.gemma_vocab_size = 262144
        self.labels = 5 # We have 5 options for MCQs
        self.out_layer = torch.nn.Linear(self.gemma_vocab_size, self.labels, device=device, dtype=torch.bfloat16)
        init.xavier_uniform_(self.out_layer.weight)
        init.zeros_(self.out_layer.bias)


    def forward(self, input_ids, targets=None):
        input_ids = input_ids.to(device)
        outputs = self.model(input_ids=input_ids)
        logits = outputs.logits
        last_logits = logits[:,-1,:] # B, 262144
        out = self.out_layer(last_logits) # B, 5

        if targets is not None:
            loss = torch.nn.functional.cross_entropy(out, targets) # B, 1
        else:
            loss = None

        return out, loss

    def generate(self, question, choices):
        prompt = f'<question> {question} \n' + f'<options>\n'
        for id, choice in enumerate(choices):
            prompt = prompt + f'{self.ID_TO_LABEL[id]}. {choice}\n'

        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        logits, _ = self.model.forward(inputs['input_ids']) # B, 5
        predicted_label = torch.argmax(logits, dim=-1)

        return self.ID_TO_LABEL[predicted_label.item()]

In [ ]:
gemma_for_mcq_model = GemmaForMCQs(model)

In [ ]:
gemma_for_mcq_model

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): Linear(in_features=1536, out_features=256, bias=False)
              (v_proj): Linear(in_features=1536, out_features=256, bias=False)
              (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
            )
            (mlp): Gemma4TextMLP(
              (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (down_pr

In [ ]:
def get_model_parameters_and_layers(model):
    parameters = 0
    layers = 0
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            parameters += module.weight.numel()
            layers += 1
        else:
            cur_parameters, cur_layers = get_model_parameters_and_layers(module)
            parameters += cur_parameters
            layers += cur_layers
    return parameters, layers

In [ ]:
print(f'{get_model_parameters_and_layers(gemma_for_mcq_model)}')

(2740518912, 527)


In [ ]:
# Freeze the model parameters.
total_params = sum(p.numel() for p in gemma_for_mcq_model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

Total trainable parameters: 5,105,608,229


In [ ]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank, dtype=torch.bfloat16, device=device))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim, dtype=torch.bfloat16, device=device))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [ ]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [ ]:
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

In [ ]:
# Freeze the model parameters.
total_params = sum(p.numel() for p in gemma_for_mcq_model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

for param in gemma_for_mcq_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in gemma_for_mcq_model.parameters() if p.requires_grad)
print(f'Total trainable parameters after freezing the model: {total_params:,}')

Total trainable parameters: 5,105,608,229
Total trainable parameters after freezing the model: 0


In [ ]:
# Unfreeze the last added output layer.
for param in gemma_for_mcq_model.out_layer.parameters():
  param.requires_grad = True

In [ ]:
replace_linear_with_lora(gemma_for_mcq_model, rank=16, alpha=16)

In [ ]:
total_params = sum(p.numel() for p in gemma_for_mcq_model.parameters() if p.requires_grad)
print(f'Total trainable parameters after applying LoRA: {total_params:,}')

Total trainable parameters after applying LoRA: 47,644,757


In [ ]:
gemma_for_mcq_model

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): LinearWithLoRA(
                (linear): Linear(in_features=1536, out_features=2048, bias=False)
                (lora): LoRALayer()
              )
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): LinearWithLoRA(
                (linear): Linear(in_features=1536, out_features=256, bias=False)
                (lora): LoRALayer()
              )
              (v_proj): LinearWithLoRA(
                (linear): Linear(in_features=1536, out_features=256, bias=False)
                (lora): LoRALayer()
              )
              (o_proj): LinearWi

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class MCQDataset(Dataset):
    LABEL_TO_ID = {
        'A': 0,
        'B': 1,
        'C': 2,
        'D': 3,
        'E': 4
    }

    def __init__(self, tokenized_list):
        self.data = tokenized_list['input_ids']
        self.labels = tokenized_list['labels']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.data[idx], dtype=torch.long, device=device),
            torch.tensor(self.LABEL_TO_ID[self.labels[idx]], dtype=torch.long, device=device)
        )

In [ ]:
from datasets import load_from_disk

common_sense_qa_dataset = load_from_disk('/content/common_sense_qa_tokenized_dataset')
common_sense_qa_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1140
    })
})

In [ ]:
print(common_sense_qa_dataset['train']['input_ids'][0])

[2, 236820, 15884, 236813, 669, 34779, 2342, 506, 2528, 964, 496, 142962, 13949, 236764, 532, 901, 10012, 531, 1144, 506, 7880, 506, 2528, 1053, 1603, 531, 2352, 236881, 236743, 107, 236820, 8194, 236813, 107, 236776, 236761, 16052, 107, 236799, 236761, 31957, 107, 236780, 236761, 92582, 107, 236796, 236761, 110954, 657, 107, 236788, 236761, 5571, 107, 1]


In [ ]:
mcq_tokenized_dataset = MCQDataset(common_sense_qa_dataset['train'])

In [ ]:
tokenizer.pad_token_id, tokenizer.decode(tokenizer.pad_token_id)

(0, '<pad>')

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def dynamic_collate_fn(batch):
    pad_token_id = tokenizer.pad_token_id

    # 1. Pad the sequences so they all match the longest one in this batch
    input_ids, target = zip(*batch)
    padded_input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)

    # 2. For Causal LM, labels are a direct copy of the input_ids
    labels = padded_input_ids.clone()

    # 3. Dynamically create the attention mask
    # It creates a tensor of 1s where the token is real, and 0s where it is padding
    attention_masks = (padded_input_ids != pad_token_id).long()

    # 4. Replace padding token ids in labels with -100 to ignore them in loss calculation
    labels[labels == pad_token_id] = -100

    # 3. Return the neat dictionary that standard NLP models expect
    return {
        'input_ids': padded_input_ids,
        'attention_mask': attention_masks,
        'labels': labels,
        'target': target
    }

In [ ]:
train_dataloader = DataLoader(
    mcq_tokenized_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=dynamic_collate_fn
)

In [ ]:
first_batch = next(iter(train_dataloader))
first_batch['input_ids'].shape, first_batch['attention_mask'].shape, first_batch['labels'].shape, first_batch['target']

(torch.Size([8, 68]),
 torch.Size([8, 68]),
 torch.Size([8, 68]),
 (tensor(2, device='cuda:0'),
  tensor(1, device='cuda:0'),
  tensor(4, device='cuda:0'),
  tensor(4, device='cuda:0'),
  tensor(0, device='cuda:0'),
  tensor(3, device='cuda:0'),
  tensor(2, device='cuda:0'),
  tensor(2, device='cuda:0')))

In [ ]:

print(tokenizer.decode(first_batch['input_ids'].tolist()[0]))
print('\n')
print(tokenizer.decode([label if label != -100 else tokenizer.pad_token_id for label in first_batch['labels'].tolist()[0]]))

<bos><question> They served wine and cheese after everybody saw the exhibits and installations where? 
<options>
A. bar
B. cemetary
C. art show
D. spaghetti sauce
E. liquor store
<eos><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


<bos><question> They served wine and cheese after everybody saw the exhibits and installations where? 
<options>
A. bar
B. cemetary
C. art show
D. spaghetti sauce
E. liquor store
<eos><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


In [ ]:
gemma_for_mcq_model.to(device)

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): LinearWithLoRA(
                (linear): Linear(in_features=1536, out_features=2048, bias=False)
                (lora): LoRALayer()
              )
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): LinearWithLoRA(
                (linear): Linear(in_features=1536, out_features=256, bias=False)
                (lora): LoRALayer()
              )
              (v_proj): LinearWithLoRA(
                (linear): Linear(in_features=1536, out_features=256, bias=False)
                (lora): LoRALayer()
              )
              (o_proj): LinearWi

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

# Standard learning rate for fine-tuning
num_epochs = 1
learning_rate = 2e-5
optimizer = AdamW(gemma_for_mcq_model.parameters(), lr=learning_rate)

# Calculate total training steps
num_training_steps = num_epochs * common_sense_qa_dataset['train'].num_rows

# Set up the learning rate scheduler
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps), desc="Training")
gemma_for_mcq_model.train() # Put model in training mode

loss_intervals = 100
losses_in_interval = []

losses = []
iterations = 0

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # 1. Forward pass
        input_ids = batch['input_ids']
        targets = batch['target']
        targets = torch.tensor([target.item() for target in targets], device=device)

        logits, loss = gemma_for_mcq_model(input_ids, targets)
        losses_in_interval.append(loss.item())

        if iterations % 100 == 0 and losses_in_interval:
          avg_loss = sum(losses_in_interval) / len(losses_in_interval)
          print(f'Loss: {avg_loss}')
          losses_in_interval = []


        # 2. Backward pass
        loss.backward()
        losses.append(loss)

        # 3. Update weights and learning rate
        optimizer.step()
        lr_scheduler.step()

        # 4. Clear gradients for the next step
        optimizer.zero_grad()

        # Update progress bar and print loss
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        iterations += 1

Training:   0%|          | 0/9741 [00:00<?, ?it/s]

Loss: 1.78125
Loss: 2.1578125
Loss: 1.786875
Loss: 1.7740625
Loss: 1.875390625
Loss: 1.8090625
Loss: 1.5750390625
Loss: 1.18474609375
Loss: 0.98177734375
Loss: 0.880615234375
Loss: 0.7502099609375
Loss: 0.859638671875
Loss: 0.864677734375


### Best Loss: 0.899

#### Iteration 1:
  - Epochs: 1
  - Learning Rate: 5e-5
  - Loss: ~2

### Iteration 2:
  - Epochs: 2
  - Learning Rate: 2e-5
  - Loss: ~0.87

In [ ]:
mcq_validation_dataset = MCQDataset(common_sense_qa_dataset['validation'])

In [ ]:
print(mcq_validation_dataset[0])
print(len(mcq_validation_dataset))

(tensor([     2, 236820,  15884, 236813,    562,  80763,   5232,    563,  13139,
           573,   1156,   5281,   4301, 236764,    840,    625,    992,  14736,
           618,    496,   5052,   4113,    657,    496,   1144, 236881, 236743,
           107, 236820,   8194, 236813,    107, 236776, 236761,   4856,    107,
        236799, 236761,   9427,    107, 236780, 236761,   7920,   4762,    107,
        236796, 236761,  34508,    107, 236788, 236761,    861,  40158,    107,
             1], device='cuda:0'), tensor(0, device='cuda:0'))
1221


In [ ]:
def get_accuracy(dataset):
  correct = 0
  total = 0
  log_interval = 100

  for row in dataset:
    #print(row)
    inputs = row[0].view(1, -1)
    target = row[1]
    gemma_for_mcq_model.eval()

    with torch.no_grad():
      logits, _ = gemma_for_mcq_model.forward(inputs)
      predicted_label = torch.argmax(logits, dim=-1)
      #print(logits)
      #print(predicted_label.item(), target.item())

    if predicted_label.item() == target:
      correct += 1

    if total % log_interval == 0 and total > 0:
      print(f'Steps: {total} Accuracy: {(correct / total) * 100.0}')

    total += 1

  print(f'Accuracy: {(correct / total) * 100.0}')

In [ ]:
get_accuracy(mcq_validation_dataset)

Steps: 100 Accuracy: 69.0
Steps: 200 Accuracy: 69.0
Steps: 300 Accuracy: 68.66666666666667
Steps: 400 Accuracy: 67.5
Steps: 500 Accuracy: 67.4
Steps: 600 Accuracy: 68.66666666666667
Steps: 700 Accuracy: 68.42857142857143
Steps: 800 Accuracy: 68.75
Steps: 900 Accuracy: 69.44444444444444
Steps: 1000 Accuracy: 69.6
Steps: 1100 Accuracy: 70.0909090909091
Steps: 1200 Accuracy: 70.08333333333333
Accuracy: 70.10647010647011


### Validation Set best accuracy: 69.94

In [ ]:
from collections import Counter

def check_prediction_distribution(dataset, num_samples=100):
    predictions = []
    actuals = []
    gemma_for_mcq_model.eval()

    print(f'Checking distribution for {num_samples} samples...')
    for i in range(min(num_samples, len(dataset))):
        row = dataset[i]
        inputs = row[0].view(1, -1)
        target = row[1].item()

        with torch.no_grad():
            logits, _ = gemma_for_mcq_model.forward(inputs)
            pred = torch.argmax(logits, dim=-1).item()
            predictions.append(pred)
            actuals.append(target)

    print(f'Predicted label counts: {Counter(predictions)}')
    print(f'Actual label counts:    {Counter(actuals)}')

check_prediction_distribution(mcq_validation_dataset)

Checking distribution for 100 samples...
Predicted label counts: Counter({2: 100})
Actual label counts:    Counter({3: 23, 4: 23, 1: 20, 0: 17, 2: 17})


In [ ]:
def get_lora_layers(model):
  lora_layers = []
  for name, module in model.named_children():
    if isinstance(module, LoRALayer):
      lora_layers.append(module)
    else:
      lora_layers.extend(get_lora_layers(module))

  return lora_layers

In [ ]:
finetuned_lora_layers = get_lora_layers(gemma_for_mcq_model)

In [ ]:
print(len(finetuned_lora_layers))
finetuned_lora_layers[0].A

527


Parameter containing:
tensor([[-0.1279,  0.1953,  0.1621,  ...,  0.1582,  0.0825, -0.0723],
        [-0.0264,  0.0491, -0.0771,  ...,  0.1211,  0.0297, -0.0128],
        [ 0.0374, -0.1777, -0.1758,  ...,  0.1904,  0.0884, -0.1660],
        ...,
        [ 0.2148,  0.0698, -0.1118,  ..., -0.0913,  0.0371, -0.2021],
        [ 0.2139,  0.2188, -0.1953,  ...,  0.1465,  0.0532,  0.1895],
        [ 0.0121,  0.0233, -0.0439,  ...,  0.1641,  0.2334, -0.1377]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)

In [ ]:
import torch.nn as nn

class LoRALayerCollection(nn.Module):
    def __init__(self, lora_layers, out_layer):
        super().__init__()
        # Proper registration for serialization
        self.lora_layers = nn.ModuleList(lora_layers)
        self.out_layer = out_layer


lora_layers_collection = LoRALayerCollection(finetuned_lora_layers, gemma_for_mcq_model.out_layer)

In [ ]:
lora_layers_collection

LoRALayerCollection(
  (lora_layers): ModuleList(
    (0-526): 527 x LoRALayer()
  )
  (out_layer): LinearWithLoRA(
    (linear): Linear(in_features=262144, out_features=5, bias=True)
    (lora): LoRALayer()
  )
)

In [ ]:
torch.save(lora_layers_collection.state_dict(), 'lora_layers_collection.pth')